## **Phase 4: Data Modelling**
This step is to convert the cleaned files into a consistent analytical model.

In [22]:
# Importing the necessary libraries
import pandas as pd
import fastf1 as f1
from pathlib import Path
import os

### Dimension Tables

#### 1. Dim Driver

In [23]:
# Creating a dictionary where the data will be fetched from results data as driver ID to be the key and a list of values as their value.
# The dictionary will make sure that the values aren't repeated.
# Once the dictionary is created, the dictionary will be saved in a Pandas dataframe as Dim_Driver

directory_folder = "f1_cleaned_data"
driver_dict = {}

# Step 1: Fetching the values from each tables into a dictionary
for file_path in Path(directory_folder).glob("202*/Round_*/*"):
    results_file = f"{file_path}/results_cleaned.parquet"
    df = pd.read_parquet(results_file)
    for index, row in df.iterrows():
        full_name_split = row['FullName'].split(" ")
        first_name = " ".join(full_name_split[:-1])
        last_name = full_name_split[-1]
        if row['Abbreviation'] not in driver_dict:
            driver_dict[row['Abbreviation']] = [
                row['DriverNumber'], row['DriverId'], first_name, last_name, row['FullName'], row['CountryCode']
            ]

# Step 2: For 2025 season, the CountryCode is not there in the table. To fix that, a manual dictionary is added to fetch the CountryCode of the drivers who got in F1 in 2025.
country_code_dict_2025 = {'NOR': 'GBR', 'VER': 'NED', 'RUS': 'GBR', 'ANT': 'ITA', 'ALB': 'THA', 'STR': 'CAN', 'HUL': 'GER', 'LEC': 'MON', 'PIA': 'AUS', 'HAM': 'GBR', 'GAS': 'FRA', 'TSU': 'JPN', 'OCO': 'FRA', 'BEA': 'GBR','LAW': 'NZL', 'BOR': 'BRA', 'ALO': 'ESP', 'SAI': 'ESP', 'DOO': 'AUS', 'HAD': 'FRA', 'COL': 'ARG'}

# Step 3: Adding the 2025 drivers' CountryCode into the original dictionary
for k1 in driver_dict.keys():
    for k2 in country_code_dict_2025.keys():
        if k1==k2:
            driver_dict[k1][-1] = country_code_dict_2025[k2]

# Step 4: Converting the dictionary into the Pandas Dataframe
columns = ["driver_id", "driver_number", "driver_code", "first_name", "last_name", "full_name", "country_code"]
dim_driver = pd.DataFrame(
    [[key, *values] for key, values in driver_dict.items()], columns=columns
)
dim_driver

,driver_id,driver_number,driver_code,first_name,last_name,full_name,country_code
0,SAI,55,sainz,Carlos,Sainz,Carlos Sainz,ESP
1,PER,11,perez,Sergio,Perez,Sergio Perez,MEX
2,HAM,44,hamilton,Lewis,Hamilton,Lewis Hamilton,GBR
3,LEC,16,leclerc,Charles,Leclerc,Charles Leclerc,MON
4,ALO,14,alonso,Fernando,Alonso,Fernando Alonso,ESP
5,NOR,4,norris,Lando,Norris,Lando Norris,GBR
6,VER,1,verstappen,Max,Verstappen,Max Verstappen,NED
7,MSC,47,schumacher,Mick,Schumacher,Mick Schumacher,GER
8,VET,5,vettel,Sebastian,Vettel,Sebastian Vettel,GER
9,MAG,20,magnussen,Kevin,Magnussen,Kevin Magnussen,DEN


#### 2. Dim Constructor

In [24]:
# We will use the same technique which was used to create the dim_driver.
# TeamID will be used as constructor_id and a hardcoded dictionary will be used to provide the constructor's nationality

directory_folder = "f1_cleaned_data"
constructor_dict = {}

# Step 1: Fetching the values from each tables into a dictionary
for file_path in Path(directory_folder).glob("202*/Round_*/*"):
    results_file = f"{file_path}/results_cleaned.parquet"
    df = pd.read_parquet(results_file)
    for index, row in df.iterrows():
        if row['TeamId'] not in constructor_dict:
            constructor_dict[row['TeamId']] = [row['TeamName'], row['TeamColor'], "constructor_nationality"]

# Step 2: A manual dictionary to add nationality of each constructors.
constructor_nationality = {'alpine': 'France', 'aston_martin': 'United Kingdom', 'ferrari': 'Italy', 'haas': 'United States', 'mclaren': 'United Kingdom', 'mercedes': 'Germany', 'rb': 'Italy', 'red_bull': 'Austria', 'williams': 'United Kingdom', 'sauber': 'Switzerland', 'alfa': 'Switzerland', 'alphatauri': 'Italy'}

# Step 3: Adding the Nationality to the constructors
for k1 in constructor_dict.keys():
    for k2 in constructor_nationality.keys():
        if k1 == k2:
            constructor_dict[k1][-1] = constructor_nationality[k2]

# Step 4: Converting the dictionary into the Pandas Dataframe
columns = ['constructor_id', 'constructor_name', 'constructor_color', 'nationality']
dim_constructor = pd.DataFrame(
    [[key, *values] for key, values in constructor_dict.items()], columns=columns
)

# Step 5: create a standardized name column in dim_constructor
dim_constructor['standardised_name'] = dim_constructor['constructor_name'].str.replace(" ", "_")
con_stan = {'Haas_F1_Team': 'Haas', 'RB': 'Racing_Bulls'}
dim_constructor['standardised_name'] = dim_constructor['standardised_name'].map(con_stan).fillna(dim_constructor['standardised_name'])

dim_constructor


,constructor_id,constructor_name,constructor_color,nationality,standardised_name
0,ferrari,Ferrari,ed1c24,Italy,Ferrari
1,red_bull,Red Bull Racing,1e5bc6,Austria,Red_Bull_Racing
2,mercedes,Mercedes,6cd3bf,Germany,Mercedes
3,alpine,Alpine,2293d1,France,Alpine
4,mclaren,McLaren,f58020,United Kingdom,McLaren
5,haas,Haas F1 Team,b6babd,United States,Haas
6,aston_martin,Aston Martin,2d826d,United Kingdom,Aston_Martin
7,williams,Williams,37bedd,United Kingdom,Williams
8,alphatauri,AlphaTauri,4e7c9b,Italy,AlphaTauri
9,alfa,Alfa Romeo,b12039,Switzerland,Alfa_Romeo


#### 3. Dim Circuit

In [25]:
# This part will be mostly manual work as we don't have the feature from fast F1 to get the circuit details
# Fetching the circuits from event name
directory_folder = "f1_cleaned_data"
events_list = []
for file_path in Path(directory_folder).glob("202*/Round_*/*"):
    year = file_path.parts[-3]
    round_folder = file_path.parts[-2]
    round_name = "_".join(round_folder.split("_")[2:])
    if round_name not in events_list:
        events_list.append(round_name)

In [26]:
# After fetching the values from the round_names, we get the below dictionary
f1_circuits_dict = {
    "silverstone": ["Silverstone Circuit", "United Kingdom", "Silverstone",5.891, 18, "Permanent"],
    "red_bull_ring": ["Red Bull Ring", "Austria", "Spielberg", 4.326, 10, "Permanent"],
    "paul_ricard": ["Circuit Paul Ricard", "France", "Le Castellet", 5.842, 15,"Permanent"],
    "hungaroring": ["Hungaroring", "Hungary", "Mogyoród", 4.381, 14, "Permanent"],
    "spa": ["Circuit de Spa-Francorchamps", "Belgium", "Stavelot", 7.004, 19, "Permanent"],
    "zandvoort": ["Circuit Zandvoort", "Netherlands","Zandvoort", 4.259, 14,"Permanent"],
    "monza": ["Autodromo Nazionale di Monza","Italy","Monza",5.793,11,"Permanent",],
    "marina_bay": ["Marina Bay Street Circuit","Singapore","Singapore",4.927,19,"Street"],
    "suzuka": ["Suzuka International Racing Course","Japan","Suzuka",5.807,18,"Permanent"],
    "cota": ["Circuit of the Americas","United States","Austin",5.513,20,"Permanent",],
    "bahrain": ["Bahrain International Circuit","Bahrain","Sakhir",5.412,15,"Permanent"],
    "rodriguez": ["Autódromo Hermanos Rodríguez","Mexico","Mexico City",4.304,17,"Permanent"],
    "interlagos": ["Autódromo José Carlos Pace","Brazil","São Paulo",4.309,15,"Permanent",],
    "yas_marina": ["Yas Marina Circuit","United Arab Emirates","Abu Dhabi",5.281,16,"Permanent"],
    "jeddah": ["Jeddah Corniche Circuit","Saudi Arabia","Jeddah",6.174,27,"Street"],
    "albert_park": ["Albert Park Circuit","Australia","Melbourne",5.278,14,"Hybrid"],
    "imola": ["Autodromo Internazionale Enzo e Dino Ferrari","Italy","Imola",4.909,19,"Permanent"],
    "miami": ["Miami International Autodrome","United States","Miami Gardens",5.412,19,"Hybrid"],
    "catalunya": ["Circuit de Barcelona-Catalunya","Spain","Montmeló",4.657,14,"Permanent",],
    "monaco": ["Circuit de Monaco", "Monaco", "Monte Carlo", 3.337, 19, "Street"],
    "baku": ["Baku City Circuit", "Azerbaijan", "Baku", 6.003, 20, "Street"],
    "gilles_villeneuve": ["Circuit Gilles Villeneuve","Canada","Montreal",4.361,14, "Hybrid"],
    "lusail": ["Lusail International Circuit","Qatar","Lusail",5.419,16,"Permanent"],
    "las_vegas": ["Las Vegas Strip Circuit","United States","Las Vegas",6.201,17,"Street"],
    "shanghai": ["Shanghai International Circuit","China","Shanghai",5.451,16,"Permanent"],
}

In [27]:
# Creating the pandas dataframe from dictionary
columns = ["circuit_id", "circuit_name", "country", "city", "circuit_length_km", "number_of_corners", "circuit_type"]
dim_circuit = pd.DataFrame(
    [[key, *values] for key, values in f1_circuits_dict.items()],
    columns=columns
)
dim_circuit

,circuit_id,circuit_name,country,city,circuit_length_km,number_of_corners,circuit_type
0,silverstone,Silverstone Circuit,United Kingdom,Silverstone,5.891,18,Permanent
1,red_bull_ring,Red Bull Ring,Austria,Spielberg,4.326,10,Permanent
2,paul_ricard,Circuit Paul Ricard,France,Le Castellet,5.842,15,Permanent
3,hungaroring,Hungaroring,Hungary,Mogyoród,4.381,14,Permanent
4,spa,Circuit de Spa-Francorchamps,Belgium,Stavelot,7.004,19,Permanent
5,zandvoort,Circuit Zandvoort,Netherlands,Zandvoort,4.259,14,Permanent
6,monza,Autodromo Nazionale di Monza,Italy,Monza,5.793,11,Permanent
7,marina_bay,Marina Bay Street Circuit,Singapore,Singapore,4.927,19,Street
8,suzuka,Suzuka International Racing Course,Japan,Suzuka,5.807,18,Permanent
9,cota,Circuit of the Americas,United States,Austin,5.513,20,Permanent


#### 4. Dim Event

In [37]:
# To get the information for Dim_Event, we need it from 2 sources, i.e., the cleaned data and Fast F1.
# First, we'll fetch the event_id, year, round_number and round_name from cleaned data. Then official_event_name, event_date, country, event_format will be fetched from FastF1 by comparing he event_name.
# Circuit Id will also be required so it can be fetched from dim_circuit by comparing the Countries from FastF1 and Dim_circuit
event_dict = {}
directory_folder = 'f1_cleaned_data'
# Step 1: Fetching the details from Cleaned Data
for file_path in Path(directory_folder).glob("202*/Round_*/*"):
    year = file_path.parts[-3]
    round_folder = file_path.parts[-2]
    round_number = round_folder.split("_")[1]
    event_id = f"{year}_{round_number}"
    round_name = " ".join(round_folder.split("_")[2:])
    if event_id not in event_dict:
        event_dict[event_id] = [year, round_number, round_name]

# Step 2: Convert the initial dictionary to Pandas Dataframe
columns = ["event_id", "season", "round_number", "event_name"]
dim_event = pd.DataFrame(
    [[key, *values] for key, values in event_dict.items()], columns=columns
)

# Step 3: Matching the values of Event_name in event_dict and Fast F1 schedule
all_schedules = []
for year in dim_event['season'].unique():
    schedule = f1.get_event_schedule(int(year), include_testing=False)
    schedule_subset = schedule[['EventName', 'OfficialEventName', 'EventDate', 'Country', 'EventFormat']].copy()
    schedule_subset.columns = ['event_name', 'official_event_name', 'event_date', 'country', 'event_format']
    schedule_subset['season'] = year
    all_schedules.append(schedule_subset)

all_schedules = pd.concat(all_schedules, ignore_index=True)
dim_event = dim_event.merge(all_schedules, on=['season', 'event_name'], how='left')

# Step 4: Get the circuit id from dim_circuit into dim_event
# Match the countries for both the tables
country_map = {
    'Great Britain': 'United Kingdom',
    'Abu Dhabi': 'United Arab Emirates',
}
dim_event['country'] = dim_event['country'].map(country_map).fillna(dim_event['country'])
# get the country and circuit_id from dim_circuit
dim_circuit_subset = dim_circuit[['circuit_id', 'country']].copy()
dim_event = dim_event.merge(dim_circuit_subset, on=['country'], how='left')

# Step 5: Map the sprint and sprint_qualifying events into one
format_map = {'sprint_shootout': 'sprint', 'sprint_qualifying': 'sprint'}
dim_event['event_format'] = dim_event['event_format'].map(format_map).fillna(dim_event['event_format'])
dim_event


,event_id,season,round_number,event_name,official_event_name,event_date,country,event_format,circuit_id
0,2022_10,2022,10,British Grand Prix,FORMULA 1 LENOVO BRITISH GRAND PRIX 2022,2022-07-03,United Kingdom,conventional,silverstone
1,2022_11,2022,11,Austrian Grand Prix,FORMULA 1 ROLEX GROSSER PREIS VON ÖSTERREICH 2022,2022-07-10,Austria,sprint,red_bull_ring
2,2022_12,2022,12,French Grand Prix,FORMULA 1 LENOVO GRAND PRIX DE FRANCE 2022,2022-07-24,France,conventional,paul_ricard
3,2022_13,2022,13,Hungarian Grand Prix,FORMULA 1 ARAMCO MAGYAR NAGYDÍJ 2022,2022-07-31,Hungary,conventional,hungaroring
4,2022_14,2022,14,Belgian Grand Prix,FORMULA 1 ROLEX BELGIAN GRAND PRIX 2022,2022-08-28,Belgium,conventional,spa
...,...,...,...,...,...,...,...,...,...
116,2025_6,2025,6,Miami Grand Prix,FORMULA 1 CRYPTO.COM MIAMI GRAND PRIX 2025,2025-05-04,United States,sprint,las_vegas
117,2025_7,2025,7,Emilia Romagna Grand Prix,FORMULA 1 AWS GRAN PREMIO DEL MADE IN ITALY E ...,2025-05-18,Italy,conventional,monza
118,2025_7,2025,7,Emilia Romagna Grand Prix,FORMULA 1 AWS GRAN PREMIO DEL MADE IN ITALY E ...,2025-05-18,Italy,conventional,imola
119,2025_8,2025,8,Monaco Grand Prix,FORMULA 1 TAG HEUER GRAND PRIX DE MONACO 2025,2025-05-25,Monaco,conventional,monaco
